모든 subj raw 데이터에 8~30Hz 주파수 대역 필터와 ICA 노이즈 제거 적용

In [ ]:
# matplotlib에서 출력한 이미지(그래프 포함)은 새 창으로 열림.
%matplotlib qt

import pandas as pd
import mne
from mne.preprocessing import ICA

""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""
# 1. 데이터 불러오고 형태 및 결측치 확인.
subj_list = ['01', '02', '03', '04', '05', '06', '07', '08', '09']
epochs_subj_list = []


for subj in subj_list : 
    file_path = rf"../dataset/A{subj}T.gdf"

    print("1-1. gdf 데이터 로딩 중--------------")
    raw_data = mne.io.read_raw_gdf(file_path, preload=True)


    df = raw_data.to_data_frame()

    print("1-2. 결측치 확인--------------")
    print(df.info())

    """"""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""
    # 2. 채널 타입 정정 및 이름 수정 후, 몽타주 적용
    # 2-1. mne 라이브러리가 자동으로 EOG 채널을 EEG 채널이라 저장해둬서 정정해야 함. -> 딕셔너리 이용
    print("1. EOG 채널 타입 정정") # -> set_chnnel_types() 사용

    eog_dict = {
        'EOG-left': 'eog',
        'EOG-central': 'eog',
        'EOG-right': 'eog'
    }

    # MNE 객체에 직업 변경 사항을 적용합니다.
    raw_data.set_channel_types(eog_dict)



    # 2-2. 10-10 시스템(Extended 10-20 system) 표준 명칭으로 이름표 교체 -> Extended 10-20 system은 다양해서 훨씬 많고 다양한 위치의 전극을 허용한다. make_standard_montage('standard_1020')를 통해서 표준 10-20 시스템 뿐만 아니라 확장된 버전도 적용 가능하다. 이것이 10-20 시스템의 장점.(확장성)
    print("2. 실험에서 채택한 10-20 시스템의 확장판에 맞게 montage 설정하기 위해 EEG 채널 이름을 표준에 맞는 이름으로 재설정") # -> rename_chnnels() 사용

    channel_rename_dict = {
        'EEG-Fz': 'Fz',
        'EEG-0': 'FC3',  'EEG-1': 'FC1',  'EEG-2': 'FCz',  'EEG-3': 'FC2',  'EEG-4': 'FC4',
        'EEG-5': 'C5',   'EEG-C3': 'C3',  'EEG-6': 'C1',   'EEG-Cz': 'Cz',  'EEG-7': 'C2',   'EEG-C4': 'C4',  'EEG-8': 'C6',
        'EEG-9': 'CP3',  'EEG-10': 'CP1', 'EEG-11': 'CPz', 'EEG-12': 'CP2', 'EEG-13': 'CP4',
        'EEG-14': 'P1',  'EEG-Pz': 'Pz',  'EEG-15': 'P2',
        'EEG-16': 'POz'
    }
    raw_data.rename_channels(channel_rename_dict)
    
    print("2. 10-20 국제 표준 두피 지도를 가져와서 데이터에 씌웁니다.")
    
    # MNE 라이브러리 안에 내장된 'standard_1020' 지도를 꺼내옵니다.
    montage = mne.channels.make_standard_montage('standard_1020')

    # 꺼내온 지도를 우리 데이터에 찰칵! 하고 덮어씌웁니다.
    raw_data.set_montage(montage)
    
    print("--- 🚜 최강의 교차 정제 파이프라인 가동 (1~30Hz 광대역 ICA) ---")

    # ==========================================
    # [트랙 A] 범인 몽타주 확보용 넓은 그물망 (1~30Hz)
    # ==========================================
    print("1. 모든 노이즈(EOG, 알파파, 근육)를 잡기 위한 1~30Hz 필터링 및 ICA 학습")
    raw_for_ica = raw_data.copy().filter(l_freq=1.0, h_freq=30.0)

    ica = ICA(n_components=15, max_iter='auto', random_state=97)
    ica.fit(raw_for_ica)

    # EOG 채널과 비교하여 1차 범인 자동 색출
    eog_indices, eog_scores = ica.find_bads_eog(raw_for_ica)
    print(f"🕵️‍♂️ 1차 자동 색출된 EOG 성분: {eog_indices}")


    # ==========================================
    # [트랙 B] AI에게 먹일 진짜 뇌파 준비 (8~30Hz)
    # ==========================================
    print("2. AI 학습을 위한 8~30Hz 타겟 뇌파 준비")
    raw_for_ai = raw_data.copy().filter(l_freq=8.0, h_freq=30.0)
    # ==========================================
    # [크로스 오버] 트랙 A의 ICA를 트랙 B에 들이받기!
    # ==========================================
    print("3. 광대역에서 찾은 모든 범인 몽타주를 8~30Hz 데이터에 적용하여 영구 제외")
    raw_clean = raw_for_ai.copy()


    # 핵심: 1~30Hz에서 똑똑하게 학습된 ica를 8~30Hz 타겟 데이터에 적용!
    ica.apply(raw_clean, exclude=eog_indices)
    # 2. CAR 적용 (전압 기준점 평탄화)
    raw_clean.set_eeg_reference('average', projection=False)

    print("--- 🎉 무결점 정제 완료! ---")


    # ------------------------------------------------------- 에포킹

    print("--- 에포킹 시작---")

    # 1. 뇌파 데이터 안에 숨겨진 '지시 타이밍(이벤트)' 추출
    events, event_dict = mne.events_from_annotations(raw_clean)
    print(f"발견된 이벤트 종류: {event_dict}")

    # 2. 데이터 싹둑 썰기 (Cue 기준 -1.0초 ~ 4.0초) -> MI는 약 4초간 지속하게 실험을 진행함.
    # baseline=(-1.0, 0.0)은 지시가 내려지기 전 1초를 '영점(기준점)'으로 잡겠다는 의미입니다.
    epochs = mne.Epochs(raw_clean, events, tmin=-1.0, tmax=4.0, event_id = event_dict, baseline=(-1.0, 0.0), preload=True, event_repeated="drop")
    # event_id는 자동적으로 1,2, ... 으로 제공되지만 이미 event_id가 따로 있다면 event_id=event_dict으로 정보를 주고, preload=True로 해당 event_dict을 event_id로 쓰게 한다.
    # event_repeated 옵션은 하나의 시간(타임 스탬프)에 여러 이벤트 플래그가 있을 때 문제를 해결하기 위한 옵션. drop과 merge가 있다.

    epochs_subj_list.append(epochs)

print("끝")

TFR로 보는 ERD/ERS

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mne.time_frequency import tfr_morlet

for i in range(9) : # subj 01~09까지
    # 0. 왼손 상상 에포크만 가져오기(데이터 설명 pdf에서 왼손 상상 에포크는 769임을 확인. + epochs.event_id로 실제로 존재함 확인.)
    epochs_left = epochs_subj_list[i]['769']

    # 1. 추적할 주파수 대역 설정 (Mu 밴드를 포함한 8~30Hz)
    freqs = np.arange(8.0, 31.0, 1.0)
    n_cycles = freqs / 2.0 

    print("시간-주파수 분석(TFR) 계산 중... (수 초 소요될 수 있습니다)")
    # 2. 왼손 상상 데이터(epochs_left)의 TFR 계산 -> TFR에는  EpochsTFR과 AverageTFR이 있고, 여기선 Epochs TFR이다. -> Epochs TFR은 아래 vmin, vmax를 쓸 수 없다.
    power_left = tfr_morlet(epochs_left, freqs=freqs, n_cycles=n_cycles, use_fft=True,
                            return_itc=False, decim=3, n_jobs=1, average=True)

    # 3. ERD/ERS 시각화 (C1/C2, C3/C4 전극)
    fig, ax = plt.subplots(nrows=2, ncols=2, figsize=(20, 15)) # 2행 2열구조(1행 C1/C2, 2행 C3/C4)

    # C1/C2, C3/C4의 그래프 설정 (vmin, vmax는 Epochs TFR에서 쓸 수 없다.)

    power_left.plot(picks=['C1'], baseline=(-1.0, 0), mode='percent', 
                    axes=ax[0][0], title=f"[subj0{i+1}]ERD/ERS Map: Left Hand MI at C1", 
                    show=False) 
    ax[0][0].axvline(0, color='black', linestyle='--', linewidth=2) # 지시 시점(0초) 기준선

    power_left.plot(picks=['C2'], baseline=(-1.0, 0), mode='percent', 
                    axes=ax[0][1], title=f"[subj0{i+1}]ERD/ERS Map: Left Hand MI at C2", 
                    show=False) 
    ax[0][1].axvline(0, color='black', linestyle='--', linewidth=2)



    power_left.plot(picks=['C3'], baseline=(-1.0, 0), mode='percent', 
                    axes=ax[1][0], title=f"[subj0{i+1}]ERD/ERS Map: Left Hand MI at C3", 
                    show=False) 
    ax[1][0].axvline(0, color='black', linestyle='--', linewidth=2)


    power_left.plot(picks=['C4'], baseline=(-1.0, 0), mode='percent', 
                    axes=ax[1][1], title=f"[subj0{i+1}]ERD/ERS Map: Left Hand MI at C4", 
                    show=False) 

    ax[1][1].axvline(0, color='black', linestyle='--', linewidth=2)



    plt.tight_layout()
    plt.show()



RESP value 기반 topo map으로 source 확인

In [ ]:
# matplotlib에서 출력한 이미지(그래프 포함)은 새 창으로 열림.
%matplotlib qt

import mne
import numpy as np
import matplotlib.pyplot as plt
from mne.time_frequency import tfr_morlet





def RESP_topoMap(epochs, title="ERSP topoMap of LH or RH") : 

    # 1. Time-Frequency 분석 (Morlet wavelets 사용)
    freqs = np.arange(8, 31, 1)  # 분석할 주파수 대역 (8Hz ~ 30Hz)
    n_cycles = freqs / 2.        # 주파수별 cycle 수 조정

    # return_itc=False로 설정하여 Power(전력) 값만 가져옵니다.
    power = tfr_morlet(epochs, freqs=freqs, n_cycles=n_cycles, 
                    use_fft=True, return_itc=False, average=True)

    # 3. Baseline Correction 적용하여 ERSP 계산
    # 자극 이전(-0.5초 ~ 0초)을 베이스라인으로 삼아 logratio를 구하고 10을 곱해 ERSP를 db(데시벨) 스케일로 구합니다.
    power.apply_baseline(mode='logratio', baseline=(-0.5, 0))
    power.data *= 10
    
    # 4. 특정 주파수 대역과 시간 구간에 대한 Topo map 시각화
    # 예: Alpha 대역 (8~12Hz), 자극 후 0.1~0.3초 사이의 평균 ERSP 지형도
    fig, ax = plt.subplots(figsize=(6, 5))
    power.plot_topomap(tmin=0.5, tmax=4.0, fmin=8, fmax=12, 
                    axes=ax, show=True, 
                    vlim=(None, None), # 컬러바 스케일 자동 지정 (수동 지정 시 (-1, 1) 등 입력)
                    )
    
    ax.set_title(title)
    
    
    
    
    
    
def RESP_topoMap_LH_RH(epochs, subj='unknown') : 
    # 0. epochs에서 왼손, 오른손 상상 이벤트의 에포크만 추출
    epochs_left = epochs['769'] # left hand MI
    epochs_right = epochs['770'] # right hand MI
        
    RESP_topoMap(epochs_left, title=f"subj {subj}'s ERSP tomoMap of LH")
    RESP_topoMap(epochs_right, title=f"subj {subj}'s ERSP tomoMap of RH")
    
    
    
for i in range(9) : 
    print(f"subj 0{i+1}'s ERSP topoMap")
    RESP_topoMap_LH_RH(epochs_subj_list[i], subj=i+1)

data leakeage를 방지하는 csp + 선형 분류 모델 파이프 라인(LDA, SVM)

In [ ]:
# matplotlib에서 출력한 이미지(그래프 포함)은 새 창으로 열림.
%matplotlib qt

from mne.decoding import CSP
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt



def csp_to_ML(epoch):
 
    # [CSP 알고리즘을 이용한 ML의 데이터 준비]
    # 1. 핀셋 타겟팅: 왼손(769)과 오른손(770) 상상 데이터만 추출
    epochs_mi = epoch['769', '770'] 
    epochs_mi_eeg = epochs_mi.copy().pick_types(eeg=True, eog=False)
    # 2. 순수 EEG 채널 중에서도 앞/뒤 뇌파를 잘라내고 '운동 피질' 주변만 핀셋 추출
    # C(중심엽), FC(전두중심엽), CP(중심두정엽) 라인의 전극만 선택하여 모델의 시야를 강제로 좁힙니다.
    motor_channels = ['FC3', 'FC1', 'FCz', 'FC2', 'FC4', 
                        'C5', 'C3', 'C1', 'Cz', 'C2', 'C4', 'C6', 
                        'CP3', 'CP1', 'CPz', 'CP2', 'CP4']

    # 기존 22개 차원에서 17개 차원으로 축소
    epochs_mi_motor = epochs_mi_eeg.copy().pick_channels(motor_channels)
    # 2. 머신러닝 모델에 넣을 3차원 특징 데이터(X)와 정답 라벨(y) 분리
    X = epochs_mi_motor.get_data(copy=True) # epochs.get_data()는 (Epochs, Channels, Time) 형태의 3차원 배열을 반환합니다.
    y = epochs_mi_motor.events[:, -1] # 이벤트 배열의 마지막 열이 라벨(예: 769, 770)입니다.
    
    """"""""""""""""""""""""""""""""""""""""""""""""""""""""""""""
    # [LDA와 SVM의 cross validation]
    # 1. Pipeline 구축 (CSP와 분류기를 하나로 묶습니다)
    # 이렇게 하면 split이나 cross_val 수행 시, 훈련 데이터에만 CSP의 fit()이 적용됩니다.
    clf_lda = Pipeline([
        ('CSP', CSP(n_components=4, reg='ledoit_wolf', log=True, norm_trace=True)),
        ('LDA', LDA())
    ])

    clf_svm = Pipeline([
        ('CSP', CSP(n_components=4, reg='ledoit_wolf', log=True, norm_trace=True)),
        ('SVM', SVC(kernel='rbf'))
    ])


    # 2. LDA 베이스라인 (Train/Test 분리) - 원본 데이터 X, y를 넣고 분리합니다.
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    clf_lda.fit(X_train, y_train) # 훈련 데이터로만 CSP와 LDA가 차례대로 학습됨
    lda_pred = clf_lda.predict(X_test)
    print(f"LDA 단일 Test 정확도: {accuracy_score(y_test, lda_pred):.2f}")


    # 3. 교차 검증(K-Fold) 및 SVM 비교
    cv = KFold(n_splits=5, shuffle=True, random_state=42)

    # cross_val_score에 원본 X, y를 넘기면 내부적으로 fold를 쪼갤 때마다 CSP를 새로 학습하여 누수를 방지합니다.
    lda_scores = cross_val_score(clf_lda, X, y, cv=cv)
    svm_scores = cross_val_score(clf_svm, X, y, cv=cv)

    print(f"LDA 5-Fold 평균 정확도: {np.mean(lda_scores):.2f}")
    print(f"SVM 5-Fold 평균 정확도: {np.mean(svm_scores):.2f}")


    # 4. 혼동 행렬 시각화 (SVM 결과 기준)
    clf_svm.fit(X_train, y_train)
    svm_pred = clf_svm.predict(X_test)
    cm = confusion_matrix(y_test, svm_pred)

    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Left Hand', 'Right Hand'], yticklabels=['Left Hand', 'Right Hand'])
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.title('SVM Confusion Matrix (No Leakage)')
    plt.show()


In [ ]:
# 1. 가설 1-2 추가실험에서 저장된 subj 03, 08의 epoch 불러오기(가장 overfitting이 심함(정확도 0.99~1.0).)
import mne

epoch_03 = mne.read_epochs("../dataset/epochs/A03T-epoch.fif", preload=True)
epoch_08 = mne.read_epochs("../dataset/epochs/A08T-epoch.fif", preload=True)


csp_to_ML(epoch_03)
csp_to_ML(epoch_08)